# Getting Started with the AGI Cognitive Benchmarks Hackathon

I spent a while figuring out what this competition actually asks for, because at first glance it looks like a typical Kaggle ML comp but it really isn't. Sharing my notes in case it saves someone else the confusion.

**The short version:** you're not training a model or making predictions. You're *building evaluation benchmarks* that test whether LLMs have real cognitive abilities — things like attention, metacognition, learning, etc. The `kaggle-benchmarks` SDK is your main tool.

This notebook walks through:
- What the competition structure actually looks like
- The 5 cognitive tracks and which one might be the easiest entry point
- How the `kaggle-benchmarks` SDK works with a working example
- My take on what makes a benchmark submission score well

---

⚠️ Note: The kaggle_benchmarks SDK cells require a benchmark task notebook environment (kaggle.com/benchmarks/tasks/new) to run. This notebook is a guide — code cells marked with the SDK are for illustration only.

## 1. What You're Actually Building

The problem this competition is solving is real and kind of fascinating. Current AI evals are mostly knowledge tests — they measure what a model has memorized, not how it reasons. Think of a student who passes a history exam by memorizing the textbook. High score, but they'd be lost if you asked them to *apply* that knowledge to something new.

Google DeepMind's cognitive framework (the paper cited in the competition) breaks intelligence into five faculties:

| Track | Core Question |
|-------|---------------|
| **Learning** | Can the model acquire new skills from a few examples, not just recall training data? |
| **Metacognition** | Does the model know what it doesn't know? Is its confidence calibrated? |
| **Attention** | Can it stay focused on what matters and ignore noise? |
| **Executive Functions** | Can it plan, inhibit bad impulses, and adapt mid-task? |
| **Social Cognition** | Does it understand intent, subtext, and social context — not just words? |

Your job is to pick one, design tasks that genuinely probe it, and show that your benchmark produces a *meaningful signal* — i.e., models differ in their scores, and those differences tell you something real.

The judges don't want:
- Tasks where every model scores 0% (too hard)
- Tasks where every model scores 100% (too easy or already trained on)
- Questions that are just trivia in disguise

They want a gradient. That's actually harder to design than it sounds.

## 2. The Submission Structure

This confused me at first so let me be explicit:

```
Your Submission
├── Writeup (required)              ← the report, ~1500 words max
│   ├── Project link → Benchmark   ← your actual benchmark (built with kaggle-benchmarks)
│   ├── Public Notebook (optional)  ← this is what gets community upvotes
│   └── Media/cover image (optional)
└── The Benchmark itself
    ├── Task 1 (a kbench notebook)
    ├── Task 2
    └── Task N...
```

The **benchmark** is a collection of `kaggle-benchmarks` task notebooks that you've grouped together. Each task runs your evaluation against multiple LLMs and records the results. Kaggle then generates a leaderboard showing how each model performed.

The **writeup** is your report explaining what the benchmark tests, why it's novel, and what the results show.

The **optional public notebook** is what I'm doing here — a guide, tutorial, or analysis that the community can upvote.

**Key note on timing:** Tasks and benchmarks are set to private during the competition. After the April 16 deadline, everything goes public.

## 3. Setting Up the SDK

The `kaggle-benchmarks` library comes pre-installed when you create a task notebook from [kaggle.com/benchmarks/tasks/new](https://www.kaggle.com/benchmarks/tasks/new). But if you want to experiment locally or understand the API, here's the setup:

In [1]:
# On Kaggle this is already installed. Locally you'd do:
# pip install kaggle-benchmarks

# Let's check the version and available components
try:
    import kaggle_benchmarks as kbench
    print("kaggle_benchmarks loaded successfully")
    print(f"Available top-level attributes: {[x for x in dir(kbench) if not x.startswith('_')]}")
except ImportError:
    print("Not running in Kaggle environment — SDK not available here.")
    print("Navigate to kaggle.com/benchmarks/tasks/new to get a pre-configured notebook.")

Not running in Kaggle environment — SDK not available here.
Navigate to kaggle.com/benchmarks/tasks/new to get a pre-configured notebook.


## 4. A Minimal Working Task

The SDK uses a decorator pattern. Here's the absolute minimum to create a valid task:

In [2]:
# This is the canonical pattern from the SDK docs
# (run this inside a Kaggle benchmark task notebook)

import kaggle_benchmarks as kbench

@kbench.task(name="simple_riddle")
def solve_riddle(llm, riddle: str, answer: str):
    """Asks a riddle and checks for a keyword in the answer."""
    response = llm.prompt(riddle)
    kbench.assertions.assert_contains_regex(
        f"(?i){answer}",
        response,
        expectation="LLM should give the right answer."
    )

# Execute the task
solve_riddle.run(
    llm=kbench.llm,
    riddle="What gets wetter as it dries?",
    answer="Towel",
)

ModuleNotFoundError: No module named 'kaggle_benchmarks'

That's it. The `@kbench.task` decorator handles the scaffolding. Under the hood it:
- Sends your prompt to the LLM
- Records the full interaction (input + output)
- Runs your assertion to mark pass/fail
- Saves results in a format Kaggle uses to build the benchmark leaderboard

Now let's look at something more realistic — a task actually designed for one of the five tracks.

## 5. A Real-ish Metacognition Task

Metacognition is probably my favorite track to work on because it's genuinely hard to fake. The question is: does the model know what it doesn't know?

Here's a task design that tests **confidence calibration** — whether a model's stated certainty matches its actual accuracy:

In [ ]:
import kaggle_benchmarks as kbench
import json
import re

# A set of questions with varying difficulty
# Good calibration = model scores ~80% on questions it rates 80% confident on
CALIBRATION_QUESTIONS = [
    {"q": "What is the capital of France?", "answer": "Paris", "difficulty": "easy"},
    {"q": "In what year was the Treaty of Westphalia signed?", "answer": "1648", "difficulty": "medium"},
    {"q": "What is the boiling point of tungsten in Celsius?", "answer": "5555", "difficulty": "hard"},
    {"q": "Who wrote the novel 'The Recognitions'?", "answer": "William Gaddis", "difficulty": "hard"},
    {"q": "What does HTTP stand for?", "answer": "HyperText Transfer Protocol", "difficulty": "easy"},
]

METACOG_PROMPT = """Answer the following question. After your answer, rate your confidence 
as a percentage from 0-100 on a new line in this exact format: CONFIDENCE: <number>

Question: {question}"""


@kbench.task(name="metacognition_calibration")
def calibration_task(llm, question: str, answer: str, difficulty: str):
    """
    Tests whether a model's stated confidence is calibrated with its actual accuracy.
    A well-calibrated model should score ~X% on questions it rates X% confidence.
    """
    response = llm.prompt(METACOG_PROMPT.format(question=question))

    # Check if the answer is correct
    answer_correct = answer.lower() in response.lower()

    # Extract stated confidence
    conf_match = re.search(r'CONFIDENCE:\s*(\d+)', response)
    stated_confidence = int(conf_match.group(1)) if conf_match else None

    # We assert two things:
    # 1. The model follows the format (has a confidence rating)
    kbench.assertions.assert_contains_regex(
        r'CONFIDENCE:\s*\d+',
        response,
        expectation="Model should state confidence in the required format."
    )

    # 2. If the model says it's very confident (>90%), it should be right
    # (This is one slice of calibration — a real benchmark would aggregate across many runs)
    if stated_confidence is not None and stated_confidence > 90:
        kbench.assertions.assert_true(
            answer_correct,
            expectation=f"Model rated {stated_confidence}% confidence but got the wrong answer."
        )


# Run across all questions
for item in CALIBRATION_QUESTIONS:
    calibration_task.run(
        llm=kbench.llm,
        question=item["q"],
        answer=item["answer"],
        difficulty=item["difficulty"],
    )

## 6. Choosing Your Track — My Take

Here's some honest analysis of each track from a "what's easiest to do well" angle:

**Metacognition** — great signal, relatively novel. Most benchmarks don't test this. The calibration approach above is well-defined and produces a quantifiable result. Medium difficulty to implement.

**Attention** — easier to construct tasks for (just add noise/distractors to prompts), but the interesting question is *how much* noise before performance degrades. You'd want a systematic series of tasks with increasing distractor density.

**Executive Functions** — multi-step planning tasks are compelling but hard to score reliably. You'd need clear intermediate checkpoints to avoid subjective grading.

**Learning** — probably the hardest to do cleanly. Few-shot generalization benchmarks are tricky because you have to ensure the pattern genuinely can't be memorized from training data. Synthetic rules/languages help a lot here.

**Social Cognition** — Theory of Mind tasks (false belief, indirect speech acts) are well-studied in cognitive science and have clean operationalizations. Lots of existing psychology literature to draw from for task inspiration.

My suggestion: **Metacognition or Attention** for a first submission. Both have clear operationalizations, measurable outputs, and are likely to show meaningful variance between strong and weak models.

## 7. What Makes a Benchmark Score Well (Evaluation Criteria)

The judges are scoring on four things:

**Dataset quality & task construction (50%)** — This is the big one. Your tasks need:
- Unambiguous correct answers (no "it depends")
- Enough examples to be statistically meaningful
- Clean assertion logic that doesn't have edge case bugs

**Writeup quality (20%)** — Basically: can someone read your writeup and understand *exactly* what you measured, how, and what you found? The template they gave is a solid guide. Don't skip the "results and insights" section — this is where you show the benchmark actually reveals something new.

**Discriminatory power (15%)** — This is where most benchmarks fail. If you run it against GPT-4o, Gemini 1.5, Claude 3, and Llama 3, do you see meaningfully different scores? If yes, your benchmark is discriminatory. If not, it's either too easy, too hard, or measuring something all models have already solved.

**Community upvotes (15%)** — Only benchmark votes count here, not writeup votes. That said, a clear and shareable writeup definitely helps people find and upvote your benchmark.

## 8. A Note on Task Design Anti-Patterns

Things I've seen in early community benchmarks that probably won't score well:

1. **Knowledge questions dressed up as cognitive tests.** Asking "Who invented the telephone?" is testing recall, not any of the five faculties. The question should be fundamentally impossible to answer by pattern-matching on training data.

2. **Subjective assertions.** If your `assert_*` statement requires human judgment to evaluate, it's not a benchmark — it's a rubric. Find a way to make the scoring automatic and defensible.

3. **No baseline comparison.** A benchmark without comparative results is just a puzzle. Run at least 2-3 models to show the spread.

4. **Over-complicated prompts.** The prompt should isolate the cognitive ability you're testing, not test everything at once. If your attention task also requires multi-step planning, you've confounded the results.

5. **Too few items.** 5 questions is not a benchmark. Aim for at least 20-30 items for statistical reliability. The SDK makes it easy to loop.

## 9. Resources Worth Bookmarking

- [kaggle-benchmarks GitHub](https://github.com/Kaggle/kaggle-benchmarks) — the SDK source and examples
- [Benchmarks Cookbook](https://www.kaggle.com/code/kaggle/benchmarks-cookbook) — advanced patterns
- [Getting Started Notebook](https://www.kaggle.com/code/kaggle/benchmarks-getting-started) — official tutorial
- [DeepMind Paper](https://arxiv.org/abs/2502.12345) — the cognitive framework this competition is based on
- [Example Tasks](https://www.kaggle.com/benchmarks/tasks) — community tasks already published

The Discord linked in the competition overview is also active and the Kaggle team has been responsive to questions about the SDK.

---

Good luck everyone. This is one of the more intellectually interesting Kaggle comps in a while — designing evals is genuinely hard and I'm curious what the community comes up with. Will keep updating this notebook as I build out my own submission.

In [ ]:
# Quick summary of the competition timeline
import pandas as pd
from datetime import date

timeline = pd.DataFrame({
    'Date': [
        'March 17, 2026',
        'April 16, 2026',
        'April 17 – May 31, 2026',
        'May 22, 2026',
        'June 1, 2026'
    ],
    'Milestone': [
        'Competition Start ← we are here',
        'Final Submission Deadline',
        'Judging Period',
        'Deadline to vote on others\' benchmarks',
        'Results Announced'
    ]
})

print(timeline.to_string(index=False))
print("\nDays remaining to deadline:", (date(2026, 4, 16) - date.today()).days)